### Macro F1 vs. Weighted F1

**Macro F1** computes F1 independently for each class, then averages those 
10 scores with equal weight — so a class with only 300 test images 
(Pasture) counts exactly as much as one with 450 (AnnualCrop, Forest, etc.). 
This is the right metric for judging whether the model is fair across all 
classes, not just the common ones.

**Weighted F1** averages per-class F1 scores weighted by each class's 
number of test examples (support) — so larger classes influence the score 
more. This better reflects real-world "overall" performance if the test 
distribution matches expected deployment conditions.

In our results, macro F1 (0.9787) and weighted F1 (0.9792) are nearly 
identical (0.0005 apart) — this tells us the model performs consistently 
well across all 10 classes, including the smaller ones (Pasture, at 300 
test images), not just the larger ones. If these two metrics had diverged 
significantly, it would indicate the model was performing well on average 
only because it excelled at large classes while neglecting small ones — 
that is not the case here.

## Explainable AI — Grad-CAM Analysis

Grad-CAM was applied to the final fine-tuned ResNet18 model to visualize 
which image regions most influenced individual predictions. Grad-CAM 
shows correlation between spatial regions and the model's output score — 
it does not prove the model's reasoning process, and should be read as an 
approximate, illustrative signal rather than a definitive explanation.

### Correct, High Confidence (PermanentCrop, 100%)
The heatmap concentrates tightly on the central cluster of field parcels 
and their boundary lines, largely ignoring blurred tile edges. This is 
consistent with the model keying on field/parcel structure — a genuine 
characteristic of cultivated land — for a confident, correct prediction.

### Correct, Low Confidence (Pasture, 49.5%)
Unlike the high-confidence example, attention is split across two separate 
regions rather than one concentrated area, connected by a band that appears 
to trace a visible path/boundary in the image. This more scattered pattern 
is visually consistent with the model's low confidence, though this is 
based on a single example and should not be generalized without further 
investigation across more low-confidence cases.

### Incorrect Prediction (true PermanentCrop, predicted AnnualCrop, 53.1%)
The heatmap concentrates on a small subregion (a lighter-toned field parcel 
near the bottom of the tile) rather than the broader parcel-division pattern 
visible across the full image. This example is consistent with — though 
does not conclusively prove — the hypothesis that the model over-weighted 
a locally AnnualCrop-like region rather than considering the tile's overall 
structure, offering one plausible mechanism behind the persistent 
PermanentCrop/AnnualCrop confusion identified in Phases 4, 9, and 11.

### Caveat
These are three illustrative examples, not a systematic study. A rigorous 
claim about what the model "typically" attends to for each class would 
require aggregating Grad-CAM patterns across many examples per class, 
which is beyond this project's scope but noted as a natural extension 
in Limitations/Future Improvements.

## Phase 13 — Error Analysis

### Overall Error Rate
The final model misclassifies 84 of 4,050 test images (2.07% error rate), 
consistent with the 97.93% test accuracy reported in Phase 11.

### Errors Are Concentrated, Not Scattered
Of the 84 total errors, 61 (72.6%) occur between classes within the 
vegetation/cropland family — AnnualCrop, Forest, HerbaceousVegetation, 
Pasture, and PermanentCrop confused with one another. The remaining 23 
errors (27.4%) are spread thinly across other class pairs (largest: 
Highway/River-related, 8 instances), with no other pattern occurring 
more than 3 times.

PermanentCrop alone is involved — as either the true or predicted class — 
in 38 of the 84 errors (45.2%), making it by a wide margin the model's 
single largest source of remaining error, consistent with every prior 
analysis in this project: the RGB similarity flagged in Phase 4's EDA, 
the confusion pattern seen in Phase 9's model comparisons, and the 
Grad-CAM example in Phase 12.

### Why This Pattern Likely Exists
Several explanations, considered together rather than any single one 
claimed as definitive:

1. **Visually similar classes.** AnnualCrop, PermanentCrop, and 
   HerbaceousVegetation are all cultivated/vegetated land at overhead 
   resolution, often differing primarily in crop-cycle stage (annual vs. 
   permanent) or vegetation density — distinctions that are inherently 
   subtle in a single 64x64 snapshot with no temporal or seasonal context.
2. **Insufficient contextual information.** A single overhead image 
   captures land cover at one moment; permanent vs. annual cropland is, 
   by definition, a distinction that depends on land use *over time*, 
   not on how a single snapshot looks — meaning true separability from 
   a single image alone may have a hard ceiling regardless of model 
   architecture or training data.
3. **Possible dataset labeling limitations.** Given how visually close 
   these categories can be even to trained review, some fraction of 
   these "errors" may reflect genuine ambiguity or borderline labeling 
   decisions in the original EuroSAT annotations, rather than a model 
   deficiency. We have not conducted a formal manual audit of ground-truth 
   labels for these examples, so this remains a plausible contributing 
   factor rather than a confirmed one.

### What This Means for the Project
This is a realistic, bounded limitation rather than a fixable bug: the 
same fundamental confusion (PermanentCrop and its neighbors) persisted, 
in a shrinking but still-present form, across every model we trained — 
baseline CNN, frozen transfer learning, and fine-tuned transfer learning. 
Each step reduced this error meaningfully (see Phase 9's experiment table), 
but none eliminated it entirely, suggesting a genuine ceiling tied to the 
information available in a single RGB overhead image, not a solvable 
training deficiency.

### Secondary Pattern: Highway/River
A much smaller residual cluster (8 of 84 errors) involves Highway, River, 
and occasionally Industrial — classes sharing linear or built-structure 
visual characteristics. This was the dominant error source in the baseline 
and frozen-backbone models (Phases 7–8) and was substantially — though 
not completely — resolved by fine-tuning (Phase 9), consistent with the 
Highway/River Grad-CAM discussion in Phase 12.